# City recognition: VLM worker (Colab)

Runs one open-weight vision-language model on the exported bundle of 124
street-level grid images and writes a `responses_<model>.jsonl` to import
back into the local database. **No web search / no tools**: the model
answers from its own weights only.

**Steps:** run the cells top to bottom. Pick a T4 GPU runtime
(*Runtime → Change runtime type → T4 GPU*). Upload `bundle_city_grid.zip`
when asked. Download the results file at the end.


## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install dependencies

In [ ]:
!pip -q install "transformers>=4.49.0" accelerate bitsandbytes qwen-vl-utils pillow

## 3. Upload and unzip the bundle
Upload `bundle_city_grid.zip` (from the repo's `colab/` folder). It unzips to
`bundle_city_grid/`.

*Alternative:* put the zip on Google Drive, mount it, and `!unzip` from there.
This is handy if you re-run across sessions.


In [ ]:
import zipfile, pathlib
from google.colab import files
up = files.upload()                     # choose bundle_city_grid.zip
name = next(iter(up))
with zipfile.ZipFile(name) as z:
    z.extractall("bundle_city_grid")
print("bundle ready:", sorted(p.name for p in pathlib.Path("bundle_city_grid").iterdir()))

## 4. Configuration
`HF_ID` is the exact, pinned HuggingFace repo, the one thing to change to run
a different Qwen-family model. (Other families such as InternVL, MiniCPM-V and
Molmo have different call signatures; a separate adapter cell is added per family.)


In [ ]:
MODEL_NAME = "qwen2.5-vl-7b"                 # short name recorded in the DB
HF_ID      = "Qwen/Qwen2.5-VL-7B-Instruct"  # exact, pinned repo
FAMILY     = "qwen"
BUNDLE     = "bundle_city_grid"
OUT        = f"responses_{MODEL_NAME}.jsonl"
MAX_NEW_TOKENS = 512
LOAD_IN_4BIT   = True                       # fits the 7B on a 16 GB T4
MAX_SIDE       = 896                        # cap grid size so vision tokens
                                            # fit the T4 (896 = 32*28, Qwen patch)

## 5. Load the bundle (prompt, schema, manifest)
Self-contained: defines `BUNDLE` itself, so it runs for any model without the Qwen config cell.

In [ ]:
import json, pathlib
BUNDLE = "bundle_city_grid"
B = pathlib.Path(BUNDLE)
prompt_base = (B / "prompt_city.txt").read_text(encoding="utf-8")
meta = json.loads((B / "bundle_meta.json").read_text())
manifest = [json.loads(l) for l in (B / "manifest.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]

# Open models have no native JSON-schema output, so we ask for JSON explicitly.
JSON_SUFFIX = (
    "\n\nRespond with ONLY a single JSON object, no other text and no markdown, "
    "with exactly these fields: city (string), country (string), latitude (number), "
    "longitude (number), confidence (number between 0 and 1), cues (array of objects, "
    "each with cue_type and description), reasoning (string)."
)
effective_prompt = prompt_base + JSON_SUFFIX
print(f"{len(manifest)} points | scheme {meta['scheme']} | prompt {meta['prompt_version']}")

## 6. Load the model (4-bit)

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

kw = dict(device_map="auto")
if LOAD_IN_4BIT:
    kw["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16)
else:
    kw["torch_dtype"] = torch.bfloat16

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(HF_ID, **kw)
processor = AutoProcessor.from_pretrained(HF_ID)
HF_REVISION = getattr(model.config, "_commit_hash", "") or ""
print("loaded", HF_ID, "| revision", HF_REVISION)

## 7. Inference helpers (greedy decoding, tolerant JSON parse)
The grid is capped to `MAX_SIDE` so the vision-token count fits a 16 GB T4, and the CUDA cache is freed after each point.

In [ ]:
import time, gc
from PIL import Image
from qwen_vl_utils import process_vision_info

def extract_json(text):
    """First balanced {...} block, parsed; None if unparseable."""
    start = text.find("{")
    if start < 0:
        return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(text[start:i + 1])
                except Exception:
                    return None
    return None

def infer(image_path, prompt):
    img = Image.open(image_path).convert("RGB")
    if max(img.size) > MAX_SIDE:                 # keep vision tokens bounded
        img = img.resize((MAX_SIDE, MAX_SIDE))
    messages = [{"role": "user", "content": [
        {"type": "image", "image": img},
        {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors="pt").to(model.device)
    t0 = time.time()
    with torch.no_grad():
        gen = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    dt = int((time.time() - t0) * 1000)
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, gen)]
    out = processor.batch_decode(trimmed, skip_special_tokens=True)[0]
    del inputs, gen, trimmed
    gc.collect(); torch.cuda.empty_cache()
    return out, dt

## 8. Run over all points → `responses_<model>.jsonl`
Resumable: re-running skips points already written.

In [ ]:
done = set()
if pathlib.Path(OUT).exists():
    for l in pathlib.Path(OUT).read_text(encoding="utf-8").splitlines():
        try:
            r = json.loads(l)
            if "point_id" in r:
                done.add(r["point_id"])
        except Exception:
            pass

mode = "a" if done else "w"
with open(OUT, mode, encoding="utf-8") as fh:
    if not done:
        fh.write(json.dumps({"_meta": {
            "model_name": MODEL_NAME, "hf_id": HF_ID, "hf_revision": HF_REVISION,
            "family": FAMILY, "scheme": meta["scheme"],
            "prompt_version": meta["prompt_version"],
            "effective_prompt": effective_prompt}}) + "\n")
    for k, m in enumerate(manifest):
        if m["point_id"] in done:
            continue
        raw, dt = infer(B / m["image"], effective_prompt)
        parsed = extract_json(raw)
        if parsed is None:                       # one retry with a firmer nudge
            raw2, dt2 = infer(B / m["image"], effective_prompt + "\n\nReturn ONLY valid JSON.")
            p2 = extract_json(raw2)
            if p2 is not None:
                raw, parsed, dt = raw2, p2, dt2
        fh.write(json.dumps({"point_id": m["point_id"], "raw_text": raw,
                             "parsed": parsed, "latency_ms": dt, "tokens": None}) + "\n")
        fh.flush()
        if (k + 1) % 20 == 0:
            print(f"{k + 1}/{len(manifest)}")
print("done ->", OUT)

## 9. Download the results
Then locally: `python -m inference import-bundle-results --results responses_<model>.jsonl`

In [ ]:
from google.colab import files
files.download(OUT)

---
# Other model families

Each family below is a **single self-contained cell** that replaces the Qwen
config + load + helpers (cells 4, 6, 7). To run one:

1. run cells **1-3** (GPU, install, upload bundle) and cell **5** (load bundle),
2. run the adapter cell for the family you want,
3. run cell **8** (the run loop) and cell **9** (download).

The run loop and the results format are identical, so the local
`import-bundle-results` step is the same for every model.


## InternVL
`HF_ID` pinned to `OpenGVLab/InternVL2_5-8B` (ungated). If it OOMs on load,
switch to `OpenGVLab/InternVL2_5-4B`. Uses InternVL's dynamic-tiling
preprocessing (capped at `MAX_TILES` so it fits the T4).

**Important:** InternVL2_5's remote code needs `transformers <= 4.49`, which
conflicts with the newer version Qwen needs (cell 2). So for InternVL:
**restart the runtime, SKIP cell 2**, and run this cell (it pins the
compatible transformers itself). Run cell 5 (bundle) before it.

In [ ]:
!pip -q install "transformers==4.47.1" accelerate bitsandbytes timm einops sentencepiece

MODEL_NAME = "internvl2.5-8b"
HF_ID      = "OpenGVLab/InternVL2_5-8B"   # OOM on load? -> OpenGVLab/InternVL2_5-4B
FAMILY     = "internvl"
OUT        = f"responses_{MODEL_NAME}.jsonl"
MAX_NEW_TOKENS = 512
MAX_TILES  = 6                             # cap dynamic tiles for the T4

import json, time, gc, torch
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
from PIL import Image
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def _tf(sz):
    return T.Compose([
        T.Lambda(lambda im: im.convert("RGB")),
        T.Resize((sz, sz), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

def _closest(ar, ratios, w, h, sz):
    best, bd = (1, 1), float("inf")
    for r in ratios:
        d = abs(ar - r[0] / r[1])
        if d < bd:
            bd, best = d, r
        elif d == bd and w * h > 0.5 * sz * sz * r[0] * r[1]:
            best = r
    return best

def _dyn(image, sz=448, max_num=6, use_thumbnail=True):
    w, h = image.size
    ratios = sorted({(i, j) for n in range(1, max_num + 1) for i in range(1, n + 1)
                     for j in range(1, n + 1) if 1 <= i * j <= max_num},
                    key=lambda x: x[0] * x[1])
    tr = _closest(w / h, ratios, w, h, sz)
    tw, th, cols = sz * tr[0], sz * tr[1], (sz * tr[0]) // sz
    resized = image.resize((tw, th))
    imgs = []
    for i in range(tr[0] * tr[1]):
        box = ((i % cols) * sz, (i // cols) * sz, ((i % cols) + 1) * sz, ((i // cols) + 1) * sz)
        imgs.append(resized.crop(box))
    if use_thumbnail and len(imgs) != 1:
        imgs.append(image.resize((sz, sz)))
    return imgs

def load_image(path, sz=448, max_num=6):
    image = Image.open(path).convert("RGB")
    tf = _tf(sz)
    return torch.stack([tf(t) for t in _dyn(image, sz=sz, max_num=max_num)])

model = AutoModel.from_pretrained(
    HF_ID, torch_dtype=torch.bfloat16, trust_remote_code=True, device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                           bnb_4bit_compute_dtype=torch.bfloat16)).eval()
tokenizer = AutoTokenizer.from_pretrained(HF_ID, trust_remote_code=True, use_fast=False)
HF_REVISION = getattr(model.config, "_commit_hash", "") or ""
GEN = dict(max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
print("loaded", HF_ID, "| revision", HF_REVISION)

def extract_json(text):
    start = text.find("{")
    if start < 0:
        return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(text[start:i + 1])
                except Exception:
                    return None
    return None

def infer(image_path, prompt):
    pv = load_image(image_path, max_num=MAX_TILES).to(torch.bfloat16).cuda()
    t0 = time.time()
    # keyword args: InternVL's chat() positional order is fragile across revisions
    resp = model.chat(tokenizer=tokenizer, pixel_values=pv,
                      question="<image>\n" + prompt, generation_config=GEN)
    dt = int((time.time() - t0) * 1000)
    del pv; gc.collect(); torch.cuda.empty_cache()
    return resp, dt

## Idefics family  (native transformers, no remote code, no version pins)
Loads with the standard `AutoProcessor` + `generate`, exactly like Qwen, with
none of InternVL's remote-code fragility. Default `HF_ID` =
`HuggingFaceM4/idefics2-8b` (ungated: Mistral-7B + SigLIP), 4-bit for the T4.
For a quick light run set `HF_ID = "HuggingFaceTB/SmolVLM-Instruct"` and
`LOAD_IN_4BIT = False` (but 2B is too weak to geolocate, country ~9%).

**Run:** restart the runtime, then cells **1, 2, 3, 5**, then this cell, then
**8, 9**. Uses the newer transformers from cell 2 (no downgrade).

In [ ]:
MODEL_NAME   = "idefics2-8b"
HF_ID        = "HuggingFaceM4/idefics2-8b"   # ungated (Mistral-7B + SigLIP)
FAMILY       = "idefics"
OUT          = f"responses_{MODEL_NAME}.jsonl"
MAX_NEW_TOKENS = 512
MAX_SIDE     = 1024
LOAD_IN_4BIT = True                          # 8B fits the T4 in 4-bit

import json, time, gc, torch
from PIL import Image
from transformers import AutoProcessor, BitsAndBytesConfig
try:
    from transformers import AutoModelForImageTextToText as _AutoVLM
except ImportError:
    from transformers import AutoModelForVision2Seq as _AutoVLM

processor = AutoProcessor.from_pretrained(HF_ID, do_image_splitting=False)
kw = {}
if LOAD_IN_4BIT:
    kw["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16)
    kw["device_map"] = "auto"
else:
    kw["torch_dtype"] = torch.bfloat16
model = _AutoVLM.from_pretrained(HF_ID, **kw).eval()
if not LOAD_IN_4BIT:
    model = model.to("cuda")
HF_REVISION = getattr(model.config, "_commit_hash", "") or ""
print("loaded", HF_ID, "| revision", HF_REVISION)

def extract_json(text):
    start = text.find("{")
    if start < 0:
        return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(text[start:i + 1])
                except Exception:
                    return None
    return None

def infer(image_path, prompt):
    img = Image.open(image_path).convert("RGB")
    if max(img.size) > MAX_SIDE:
        img = img.resize((MAX_SIDE, MAX_SIDE))
    messages = [{"role": "user", "content": [
        {"type": "image"}, {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=text, images=[img], return_tensors="pt").to(model.device)
    t0 = time.time()
    with torch.no_grad():
        gen = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    dt = int((time.time() - t0) * 1000)
    out = processor.batch_decode(gen[:, inputs["input_ids"].shape[1]:],
                                 skip_special_tokens=True)[0]
    del inputs, gen; gc.collect(); torch.cuda.empty_cache()
    return out, dt